# Value iteration: one backup to rule them all

_Reinforcement Learning_

**Collapse policy iteration's two steps into a single Bellman optimality backup, and converge to V* geometrically because that backup is a contraction.**

Policy iteration alternates two phases: evaluate the current policy to convergence,
       then improve it greedily. Value iteration notices you do not need a fully evaluated policy to improve &mdash;
       one backup of evaluation, immediately followed by the greedy $\max$, is enough. Collapse the two steps into one.

       The single combined update is the Bellman optimality backup. Apply it to every state, sweep after sweep.
       The values march toward $V^*$, the optimal value function. Read off the greedy action in each state and you have $\pi^*$, the optimal policy.

---

This notebook is a practice scaffold for the **Value iteration: one backup to rule them all** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — Python + NumPy (gridworld; FrozenLake-style)

In [ ]:
import numpy as np

# 4x4 FrozenLake-style gridworld. S=start, F=frozen, H=hole, G=goal.
#   S F F F
#   F H F H
#   F F F H
#   H F F G
GRID = ["SFFF", "FHFH", "FFFH", "HFFG"]
ROWS, COLS = 4, 4
holes  = {(r, c) for r in range(ROWS) for c in range(COLS) if GRID[r][c] == "H"}
goal   = next((r, c) for r in range(ROWS) for c in range(COLS) if GRID[r][c] == "G")
states = [(r, c) for r in range(ROWS) for c in range(COLS)]
ACTS   = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}   # up, down, left, right
ARROW  = {0: "^", 1: "v", 2: "<", 3: ">"}
gamma, tol = 0.9, 1e-10

def terminal(s): return s in holes or s == goal

def step(s, a):                              # deterministic move; walls bounce back
    if terminal(s): return s
    dr, dc = ACTS[a]
    ns = (s[0] + dr, s[1] + dc)
    if not (0 <= ns[0] < ROWS and 0 <= ns[1] < COLS): ns = s
    return ns

def reward(s, ns):                           # +1 only for stepping onto the goal
    return 1.0 if (ns == goal and s != goal) else 0.0

# ----- value iteration: sweep the Bellman OPTIMALITY backup to convergence -----
V = {s: 0.0 for s in states}
for sweep in range(10_000):
    nV, delta = dict(V), 0.0
    for s in states:
        if terminal(s):                      # holes/goal keep value 0
            continue
        best = max(reward(s, step(s, a)) + gamma * V[step(s, a)] for a in ACTS)
        nV[s] = best
        delta = max(delta, abs(nV[s] - V[s]))
    V = nV
    if delta < tol:
        break
print(f"converged in {sweep + 1} sweeps (max change {delta:.2e})")

# ----- read off the greedy policy pi* from V* -----
pi = {}
for s in states:
    if terminal(s):
        pi[s] = "G" if s == goal else "H"
    else:
        pi[s] = ARROW[max(ACTS, key=lambda a: reward(s, step(s, a)) + gamma * V[step(s, a)])]

print("\nV* (optimal value per cell):")
for r in range(ROWS):
    print("  " + "  ".join(f"{V[(r, c)]:.3f}" for c in range(COLS)))
print("\npi* (greedy policy):")
for r in range(ROWS):
    print("  " + " ".join(pi[(r, c)] for c in range(COLS)))
# converged in 7 sweeps (max change 0.00e+00)
# V*:  0.590 0.656 0.729 0.656 / 0.656 0.000 0.810 0.000 / 0.729 0.810 0.900 0.000 / 0.000 0.900 1.000 0.000
# pi*: v  >  v  <  / v  H  v  H / >  v  v  H / H  >  >  G

## Visualize the data & results

_How fast does value iteration converge? Plot the max value-change per sweep — the contraction predicts geometric decay toward 0._

In [ ]:
import numpy as np

# Tiny 3x4 stochastic gridworld (no gym): goal +1, hazard -1, one wall.
ROWS, COLS = 3, 4
WALL, GOAL, HAZARD = (1, 1), (0, 3), (1, 3)
gamma, step_cost = 0.9, -0.04
acts = {"up": (-1, 0), "down": (1, 0), "left": (0, -1), "right": (0, 1)}
def ok(s): return 0 <= s[0] < ROWS and 0 <= s[1] < COLS and s != WALL
states = [(r, c) for r in range(ROWS) for c in range(COLS) if (r, c) != WALL]
def reward(s): return 1.0 if s == GOAL else (-1.0 if s == HAZARD else step_cost)
def trans(s, a):                              # 0.8 intended, 0.1 each perpendicular slip
    perp = ["left", "right"] if a in ("up", "down") else ["up", "down"]
    res = {}
    for p, mv in [(0.8, a), (0.1, perp[0]), (0.1, perp[1])]:
        d = acts[mv]; ns = (s[0] + d[0], s[1] + d[1])
        if not ok(ns): ns = s
        res[ns] = res.get(ns, 0) + p
    return res

# Sweep the Bellman optimality backup; record the max change per sweep.
V = {s: 0.0 for s in states}
deltas = []
for sweep in range(1000):
    nV, delta = {}, 0.0
    for s in states:
        if s in (GOAL, HAZARD):
            nV[s] = reward(s); continue
        nV[s] = reward(s) + gamma * max(
            sum(p * V[ns] for ns, p in trans(s, a).items()) for a in acts)
        delta = max(delta, abs(nV[s] - V[s]))
    V = nV
    deltas.append(round(delta, 6))
    if delta < 1e-8:
        break

print("sweeps:", len(deltas))
print("per-sweep ||V_{k+1}-V_k||_inf:", deltas)
# sweeps: 27
# [0.04, 0.7128, 0.506736, 0.359018, 0.253244, 0.198886, 0.123413, 0.072437,
#  0.035482, 0.016783, 0.007518, 0.003303, 0.001417, 0.000601, 0.000252,
#  0.000105, 4.3e-05, 1.8e-05, 7e-06, 3e-06, 1e-06, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
# Ratio delta[k+1]/delta[k] settles near gamma=0.9 -> the contraction, made visible.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** In a state $s$ with two actions, the backed-up values are $\sum_{s'}P(s'\mid s,a)[R+\gamma V_k(s')] = 3$ for action "up" and $7$ for action "down". What is $V_{k+1}(s)$ and what is $\pi^*(s)$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply the optimality backup: take the $\max$ over actions. — _Value iteration stores the value of the best action, not an average._
- $V_{k+1}(s) = \max(3,7) = 7$. — _The $\max$ gives the optimal number._
- $\pi^*(s) = \arg\max(3,7) = \text{down}$. — _The $\arg\max$ gives the action that attains it._

**Answer:** $V_{k+1}(s)=7$ and $\pi^*(s)=\text{down}$.

</details>

**Problem 2.** With $\gamma = 0.9$, after one sweep the worst-case error obeys $\lVert V_1 - V^*\rVert_\infty \le \gamma\,\lVert V_0 - V^*\rVert_\infty$. If the initial error $\lVert V_0 - V^*\rVert_\infty = 100$, bound the error after 3 sweeps.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use geometric decay: $\lVert V_k - V^*\rVert_\infty \le \gamma^{k}\,\lVert V_0 - V^*\rVert_\infty$. — _Each sweep is one application of the $\gamma$-contraction $T$, so error multiplies by $\gamma$._
- Plug in: $\gamma^3 = 0.9^3 = 0.729$, so the bound is $0.729 \times 100 = 72.9$. — _Three sweeps shrink the worst-case error by $\gamma^3$._

**Answer:** At most $72.9$. (Slow! That is why $\gamma$ near $1$ needs many sweeps.)

</details>

**Problem 3.** Roughly how many sweeps does value iteration need to reach tolerance $\varepsilon = 10^{-3}$ when $\gamma = 0.9$, ignoring the initial-error constant?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Solve $\gamma^{k} \le \varepsilon$ for $k$: $k \ge \log(\varepsilon)/\log(\gamma)$. — _The error is bounded by $\gamma^{k}$ times a constant; set that below $\varepsilon$._
- Compute $\log(10^{-3})/\log(0.9) = (-3)/(-0.0458) \approx 65$. — _Both logs are negative, so the ratio is positive._

**Answer:** About $65$ sweeps. Halving $\varepsilon$ adds only a constant number of sweeps; pushing $\gamma$ toward $1$ multiplies them.

</details>